# TomTom Live Traffic Features

TomTom does not replace the dataset. It adds a **present-day** signal on top of the historical
record: how congested a road is *right now* and whether anything is *actively happening* near
it. The gap between "historically risky" and "currently jammed" is itself useful.

We call exactly two endpoints, per the brief:

- **Flow Segment Data** -> `congestion_ratio = currentSpeed / freeFlowSpeed`, plus `frc`
  (road class) and `roadClosure`.
- **Incident Details** -> `live_incident_count` and `max_live_delay` inside a small box
  around each location.

Free tier is limited (Flow ~20k/mo, Incidents ~2.5k/mo), so we only enrich the **top-N by
Module 1 priority** and cache every response to disk. Output `traffic_features.json` feeds the
shared feature table and the fusion step. The key stays in this notebook; nothing ships to a
browser.

### 1. Pick the locations worth a live call (top priority)

In [ ]:
import pandas as pd
import numpy as np
import json, time, math
import urllib.request, urllib.parse

TOMTOM_KEY = "PpByAUv9vWNVT4N6UQmS85J012CLtGys"  # TomTom API key
TOP_N = 200           # stay well under the free Incident-Details quota
BOX_M = 300           # half-size (metres) of the box for the incidents call
REFRESH = False       # True = ignore cache and re-fetch
OUT_PATH = "traffic_features.json"

locations = pd.read_csv("../dataset/locations.csv")
priority = pd.read_json("module1_priority.json")
for df in (locations, priority):
    df["location_id"] = df["location_id"].astype(str).str.strip()

targets = (
    locations.merge(priority[["location_id", "priority_rank"]], on="location_id", how="inner")
    .sort_values("priority_rank").head(TOP_N).reset_index(drop=True)
)
print("enriching top", len(targets), "locations")
targets[["location_id", "road_type", "priority_rank"]].head(5)


### 2. Cached TomTom calls (Flow + Incident Details)

In [ ]:
def _get(url, retries=3):
    for a in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                return json.loads(r.read().decode())
        except urllib.error.HTTPError as e:
            if e.code == 429: time.sleep(1.5 * (a + 1)); continue
            return None
        except Exception:
            time.sleep(0.8)
    return None

def flow(lat, lon):
    base = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    qs = urllib.parse.urlencode({"point": f"{lat},{lon}", "unit": "KMPH", "key": TOMTOM_KEY})
    j = _get(f"{base}?{qs}")
    return (j or {}).get("flowSegmentData")

def incidents_near(lat, lon, box_m=BOX_M):
    dlat = box_m / 111_000.0
    dlon = box_m / (111_000.0 * math.cos(math.radians(lat)) or 1)
    bbox = f"{lon-dlon},{lat-dlat},{lon+dlon},{lat+dlat}"
    base = "https://api.tomtom.com/traffic/services/5/incidentDetails"
    fields = "{incidents{type,properties{iconCategory,magnitudeOfDelay}}}"
    qs = urllib.parse.urlencode({"bbox": bbox, "fields": fields, "key": TOMTOM_KEY})
    j = _get(f"{base}?{qs}")
    inc = (j or {}).get("incidents", []) if j else []
    delays = [i.get("properties", {}).get("magnitudeOfDelay", 0) or 0 for i in inc]
    return {"live_incident_count": len(inc), "max_live_delay": max(delays) if delays else 0}

# cache from previous export
cache = {}
if not REFRESH:
    try:
        for row in json.load(open(OUT_PATH)):
            cache[row["location_id"]] = row
    except FileNotFoundError:
        pass
print(len(cache), "cached")


### 3. Fetch and merge the live features

In [ ]:
rows = []
measured_at = pd.Timestamp.utcnow().isoformat()
for i, r in targets.iterrows():
    c = cache.get(r["location_id"])
    if c and c.get("congestion_ratio") is not None:
        rows.append(c); continue
    f = flow(r["latitude"], r["longitude"]) if TOMTOM_KEY else None
    inc = incidents_near(r["latitude"], r["longitude"]) if TOMTOM_KEY else {"live_incident_count": None, "max_live_delay": None}
    cr = None
    if f and f.get("freeFlowSpeed"):
        cr = max(0.0, min(1.0, f["currentSpeed"] / f["freeFlowSpeed"]))
    rows.append({
        "location_id": r["location_id"], "zone": r["zone"],
        "latitude": r["latitude"], "longitude": r["longitude"],
        "priority_rank": int(r["priority_rank"]), "road_type": r["road_type"],
        "currentSpeed": (f or {}).get("currentSpeed"),
        "freeFlowSpeed": (f or {}).get("freeFlowSpeed"),
        "congestion_ratio": cr,
        "frc": (f or {}).get("frc"),
        "roadClosure": bool((f or {}).get("roadClosure")) if f else False,
        "confidence": (f or {}).get("confidence"),
        "live_incident_count": inc["live_incident_count"],
        "max_live_delay": inc["max_live_delay"],
        "measured_at": measured_at,
    })
    if (i + 1) % 25 == 0: print(f"  {i+1}/{len(targets)}")

tf = pd.DataFrame(rows)
got = tf["congestion_ratio"].notna().sum()
print(f"\nlive flow for {got}/{len(tf)} | roads closed now: {int(tf['roadClosure'].sum())}")
tf[["location_id", "congestion_ratio", "roadClosure", "live_incident_count", "max_live_delay"]].head(6)


### 4. Export `traffic_features.json`

In [ ]:
tf = tf.replace([np.inf, -np.inf], np.nan)
tf.to_json(OUT_PATH, orient="records", indent=2)
print(f"Exported {len(tf)} locations -> {OUT_PATH}")
print("avg congestion_ratio (1=free-flow, 0=jammed):",
      round(float(tf["congestion_ratio"].dropna().mean()), 3))
print(tf.head(2).to_json(orient="records", indent=2))
